# 1. Abrir CSV

In [1]:
import pandas as pd
import altair as alt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
df = pd.read_csv("base_cultura_popular_limpia.csv", delimiter=";", encoding="UTF-8")
df.head()

,appearance_id,artista_proyecto,nombre_cancion,aparicion,tipo,titulo_obra,temporada_episodio,pais_obra,lanzamiento_cancion
0,BLU001,Blur,Badhead,2013,Serie,My Mad Fat Diary,S1/E4,Reino Unido,1994
1,BLU002,Blur,Beetlebum,2008,Videojuego,Rock Band,NaN,Estados Unidos,1997
2,BLU003,Blur,Beetlebum,2015,Serie,My Mad Fat Diary,S3/E2,Reino Unido,1997
3,BLU004,Blur,Beetlebum,1997,Serie,Daria,S1/E9,Estados Unidos,1997
4,BLU005,Blur,Brothers and Sisters,2011,Serie,Top Gear (UK),S17/E5,Reino Unido,2003


#2. Tamaño y estructura de la base

In [3]:
df.shape

(411, 9)

In [4]:
df.columns

Index(['appearance_id', 'artista_proyecto', 'nombre_cancion', 'aparicion',
       'tipo', 'titulo_obra', 'temporada_episodio', 'pais_obra',
       'lanzamiento_cancion'],
      dtype='object')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 411 entries, 0 to 410
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   appearance_id        411 non-null    object
 1   artista_proyecto     411 non-null    object
 2   nombre_cancion       411 non-null    object
 3   aparicion            411 non-null    int64 
 4   tipo                 411 non-null    object
 5   titulo_obra          411 non-null    object
 6   temporada_episodio   239 non-null    object
 7   pais_obra            411 non-null    object
 8   lanzamiento_cancion  411 non-null    int64 
dtypes: int64(2), object(7)
memory usage: 29.0+ KB


In [6]:
df["artista_proyecto"].unique()

array(['Blur', 'Oasis', 'Gorillaz', 'Beady Eye',
       'The Good, the Bad & the Queen', 'Liam Gallagher',
       "Noel Gallagher's High Flying Birds", 'Noel Gallagher',
       'Damon Albarn'], dtype=object)

In [7]:
df["musico"] = df["artista_proyecto"].replace({     #OJO, ESTA SE PODRÍA BORRAR
    "Blur": "Damon Albarn",
    "Gorillaz": "Damon Albarn",
    "Damon Albarn": "Damon Albarn",
    "The Good, the Bad & the Queen": "Damon Albarn",
    "Beady Eye": "Liam Gallagher",
    "Liam Gallagher": "Liam Gallagher",
    "Noel Gallagher's High Flying Birds": "Noel Gallagher",
    "Noel Gallagher": "Noel Gallagher",
    "Oasis": "Oasis (compartido)"
})

df[["artista_proyecto", "musico"]].drop_duplicates()

,artista_proyecto,musico
0,Blur,Damon Albarn
121,Oasis,Oasis (compartido)
270,Gorillaz,Damon Albarn
386,Beady Eye,Liam Gallagher
393,"The Good, the Bad & the Queen",Damon Albarn
394,Liam Gallagher,Liam Gallagher
396,Noel Gallagher's High Flying Birds,Noel Gallagher
406,Noel Gallagher,Noel Gallagher
408,Damon Albarn,Damon Albarn


In [8]:

# Agrupar los proyectos según sus músicos

df["grupo"] = df["artista_proyecto"].replace({
    "Blur": "Damon Albarn",
    "Gorillaz": "Damon Albarn",
    "The Good, the Bad & the Queen": "Damon Albarn",
    "Damon Albarn": "Damon Albarn",
    "Oasis": "Los Gallagher",
    "Beady Eye": "Los Gallagher",
    "Liam Gallagher": "Los Gallagher",
    "Noel Gallagher": "Los Gallagher",
    "Noel Gallagher's High Flying Birds": "Los Gallagher"
})

df[["artista_proyecto", "grupo"]].drop_duplicates()


,artista_proyecto,grupo
0,Blur,Damon Albarn
121,Oasis,Los Gallagher
270,Gorillaz,Damon Albarn
386,Beady Eye,Los Gallagher
393,"The Good, the Bad & the Queen",Damon Albarn
394,Liam Gallagher,Los Gallagher
396,Noel Gallagher's High Flying Birds,Los Gallagher
406,Noel Gallagher,Los Gallagher
408,Damon Albarn,Damon Albarn


In [9]:

# Contar las apariciones de cada proyecto musical

resumen = df.groupby(
    ["grupo", "artista_proyecto"]
).size().reset_index(name="apariciones")

resumen


,grupo,artista_proyecto,apariciones
0,Damon Albarn,Blur,121
1,Damon Albarn,Damon Albarn,3
2,Damon Albarn,Gorillaz,116
3,Damon Albarn,"The Good, the Bad & the Queen",1
4,Los Gallagher,Beady Eye,7
5,Los Gallagher,Liam Gallagher,2
6,Los Gallagher,Noel Gallagher,2
7,Los Gallagher,Noel Gallagher's High Flying Birds,10
8,Los Gallagher,Oasis,149


In [10]:

# Crear el gráfico de barras apiladas

grafico = alt.Chart(resumen).mark_bar().encode(
    x=alt.X("grupo:N", title="Grupo de músicos"),
    y=alt.Y("apariciones:Q", title="Número de apariciones"),
    color=alt.Color("artista_proyecto:N", title="Proyecto musical"),
    tooltip=["grupo", "artista_proyecto", "apariciones"]
).properties(
    title="La presencia en la cultura popular según los proyectos musicales",
    width=500,
    height=350
)

grafico


alt.Chart(...)

In [11]:
# Mostrar las apariciones de cada proyecto según el tipo de contenido

conteo = (
    df.groupby(["artista_proyecto", "tipo"])
    .size()
    .reset_index(name="apariciones")
)

# Ordenar los resultados por proyecto y tipo de contenido
conteo = conteo.sort_values(
    ["artista_proyecto", "tipo"]
)

# Mostrar la tabla completa
print(conteo.to_string(index=False))


                  artista_proyecto       tipo  apariciones
                         Beady Eye      Serie            6
                         Beady Eye Videojuego            1
                              Blur   Pelicula           14
                              Blur      Serie           73
                              Blur Videojuego           34
                      Damon Albarn   Pelicula            2
                      Damon Albarn      Serie            1
                          Gorillaz   Pelicula           10
                          Gorillaz      Serie           54
                          Gorillaz Videojuego           52
                    Liam Gallagher      Serie            2
                    Noel Gallagher   Pelicula            2
Noel Gallagher's High Flying Birds      Serie           10
                             Oasis   Pelicula           27
                             Oasis      Serie           92
                             Oasis Videojuego           

In [12]:

# Sumar las apariciones por grupo y tipo de contenido

df["grupo"] = df["artista_proyecto"].replace({
    "Blur": "Damon Albarn",
    "Damon Albarn": "Damon Albarn",
    "Gorillaz": "Damon Albarn",
    "The Good, the Bad & the Queen": "Damon Albarn",
    "Oasis": "Los Gallagher",
    "Beady Eye": "Los Gallagher",
    "Liam Gallagher": "Los Gallagher",
    "Noel Gallagher": "Los Gallagher",
    "Noel Gallagher's High Flying Birds": "Los Gallagher"
})

comparacion = (
    df.groupby(["grupo", "tipo"])
    .size()
    .reset_index(name="apariciones")
)

print(comparacion.to_string(index=False))


        grupo       tipo  apariciones
 Damon Albarn   Pelicula           26
 Damon Albarn      Serie          129
 Damon Albarn Videojuego           86
Los Gallagher   Pelicula           29
Los Gallagher      Serie          110
Los Gallagher Videojuego           31


In [13]:

# Asociar cada proyecto con su grupo musical

df["grupo"] = df["artista_proyecto"].replace({
    "Blur": "Damon Albarn",
    "Damon Albarn": "Damon Albarn",
    "Gorillaz": "Damon Albarn",
    "The Good, the Bad & the Queen": "Damon Albarn",
    "Oasis": "Los Gallagher",
    "Beady Eye": "Los Gallagher",
    "Liam Gallagher": "Los Gallagher",
    "Noel Gallagher": "Los Gallagher",
    "Noel Gallagher's High Flying Birds": "Los Gallagher"
})

# Contar las apariciones por grupo y tipo de contenido

comparacion = (
    df.groupby(["grupo", "tipo"])
    .size()
    .reset_index(name="apariciones")
)

print(comparacion.to_string(index=False))


        grupo       tipo  apariciones
 Damon Albarn   Pelicula           26
 Damon Albarn      Serie          129
 Damon Albarn Videojuego           86
Los Gallagher   Pelicula           29
Los Gallagher      Serie          110
Los Gallagher Videojuego           31


In [14]:

import pandas as pd
import altair as alt


In [15]:

grafico2 = alt.Chart(comparacion).mark_bar().encode(
    x=alt.X(
        "tipo:N",
        title="Tipo de contenido",
        sort=["Serie", "Pelicula", "Videojuego"]
    ),
    y=alt.Y(
        "apariciones:Q",
        title="Número de apariciones"
    ),
    xOffset=alt.XOffset("grupo:N"),
    color=alt.Color(
        "grupo:N",
        title="Grupo musical",
        scale=alt.Scale(
            domain=["Damon Albarn", "Los Gallagher"],
            range=["#8064A2", "#4776A8"]
        )
    ),
    tooltip=["grupo", "tipo", "apariciones"]
).properties(
    title="Dos legados del Britpop: música en series, películas y videojuegos",
    width=500,
    height=350
)

grafico2


alt.Chart(...)

In [16]:
import pandas as pd
import altair as alt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [17]:
df = pd.read_csv("base_cultura_popular_limpia.csv", delimiter=";", encoding="UTF-8")
df.head()

,appearance_id,artista_proyecto,nombre_cancion,aparicion,tipo,titulo_obra,temporada_episodio,pais_obra,lanzamiento_cancion
0,BLU001,Blur,Badhead,2013,Serie,My Mad Fat Diary,S1/E4,Reino Unido,1994
1,BLU002,Blur,Beetlebum,2008,Videojuego,Rock Band,NaN,Estados Unidos,1997
2,BLU003,Blur,Beetlebum,2015,Serie,My Mad Fat Diary,S3/E2,Reino Unido,1997
3,BLU004,Blur,Beetlebum,1997,Serie,Daria,S1/E9,Estados Unidos,1997
4,BLU005,Blur,Brothers and Sisters,2011,Serie,Top Gear (UK),S17/E5,Reino Unido,2003


In [18]:
import pandas as pd
import altair as alt

# Convertir el año de aparición a un valor numérico
df["aparicion"] = pd.to_numeric(df["aparicion"], errors="coerce")

# Eliminar registros sin año de aparición o artista
datos = df.dropna(subset=["aparicion", "artista_proyecto"]).copy()

# Contar las apariciones por año y artista/proyecto
distribucion = (
    datos.groupby(["aparicion", "artista_proyecto"])
    .size()
    .reset_index(name="cantidad_apariciones")
)

# Crear el gráfico
grafico = alt.Chart(distribucion).mark_line(
    point=True
).encode(
    x=alt.X(
        "aparicion:O",
        title="Año de aparición",
        axis=alt.Axis(labelAngle=-45)
    ),
    y=alt.Y(
        "cantidad_apariciones:Q",
        title="Cantidad de apariciones",
        scale=alt.Scale(zero=True)
    ),
    color=alt.Color(
        "artista_proyecto:N",
        title="Artista o proyecto"
    ),
    tooltip=[
        alt.Tooltip("aparicion:O", title="Año"),
        alt.Tooltip("artista_proyecto:N", title="Artista o proyecto"),
        alt.Tooltip("cantidad_apariciones:Q", title="Apariciones")
    ]
).properties(
    title="Distribución temporal de apariciones por artista o proyecto",
    width=700,
    height=400
).interactive()

grafico

alt.Chart(...)

In [19]:

import pandas as pd
import altair as alt

# Asociar cada proyecto con su grupo musical
df["grupo"] = df["artista_proyecto"].replace({
    "Blur": "Damon Albarn",
    "Damon Albarn": "Damon Albarn",
    "Gorillaz": "Damon Albarn",
    "The Good, the Bad & the Queen": "Damon Albarn",
    "Oasis": "Los Gallagher",
    "Beady Eye": "Los Gallagher",
    "Liam Gallagher": "Los Gallagher",
    "Noel Gallagher": "Los Gallagher",
    "Noel Gallagher's High Flying Birds": "Los Gallagher"
})

# Convertir el año de aparición a formato numérico
df["aparicion"] = pd.to_numeric(df["aparicion"], errors="coerce")

# Conservar registros con año y grupo identificados
datos = df.dropna(subset=["aparicion", "grupo"]).copy()

# Contar las apariciones por año y grupo
distribucion = (
    datos.groupby(["aparicion", "grupo"])
    .size()
    .reset_index(name="cantidad_apariciones")
)

# Crear el gráfico de líneas
grafico = alt.Chart(distribucion).mark_line(
    point=True
).encode(
    x=alt.X(
        "aparicion:O",
        title="Año de aparición",
        axis=alt.Axis(labelAngle=-45)
    ),
    y=alt.Y(
        "cantidad_apariciones:Q",
        title="Cantidad de apariciones",
        scale=alt.Scale(zero=True)
    ),
    color=alt.Color(
        "grupo:N",
        title="Grupo",
        sort=["Damon Albarn", "Los Gallagher"]
    ),
    tooltip=[
        alt.Tooltip("aparicion:O", title="Año"),
        alt.Tooltip("grupo:N", title="Grupo"),
        alt.Tooltip("cantidad_apariciones:Q", title="Apariciones")
    ]
).properties(
    title="Distribución temporal de apariciones por grupo musical",
    width=700,
    height=400
).interactive()

grafico


alt.Chart(...)